# **Project Setup & Imports**

In [2]:
import pandas as pd
import numpy as np
from scipy.io import arff
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

***There was a problem with the original data set 'chronic_kidney_disease.arff' that I was working with.***

**What was wrong: the original file has a few small typos from whoever transcribed it — one row had a stray space (yes, yes), and two others had extra/misplaced commas that pushed the field count to 26 instead of 25. scipy's ARFF parser has zero tolerance for this**

***So I cleaned it up since it woulsn't work otherwis.***

**What I did: stripped whitespace around every value and removed the erroneous extra commas, restoring each row to the correct 25 fields (24 attributes + class). Verified it now loads to a 400×25 DataFrame with the class split matching the docs (250 ckd / 150 notckd).**

Here's the full breakdown of all 25 columns in `chronic_kidney_disease_clean.arff` — 24 clinical features + the target class.

**Demographics / vitals (numeric)**

| Column | Meaning | Units / range |
|---|---|---|
| `age` | Patient age | years |
| `bp` | Blood pressure | mm/Hg |

**Urinalysis (nominal/categorical, despite looking numeric)**

| Column | Meaning | Values |
|---|---|---|
| `sg` | Specific gravity (urine concentration) | 1.005, 1.010, 1.015, 1.020, 1.025 |
| `al` | Albumin (protein in urine — key CKD marker) | 0–5 |
| `su` | Sugar (glucose in urine) | 0–5 |
| `rbc` | Red blood cells in urine | normal / abnormal |
| `pc` | Pus cell | normal / abnormal |
| `pcc` | Pus cell clumps | present / notpresent |
| `ba` | Bacteria | present / notpresent |

**Blood chemistry (numeric)**

| Column | Meaning | Units |
|---|---|---|
| `bgr` | Blood glucose, random | mg/dL |
| `bu` | Blood urea | mg/dL |
| `sc` | Serum creatinine (key kidney-function marker) | mg/dL |
| `sod` | Sodium | mEq/L |
| `pot` | Potassium | mEq/L |
| `hemo` | Hemoglobin | g/dL |
| `pcv` | Packed cell volume (hematocrit) | % |
| `wbcc` | White blood cell count | cells/cumm |
| `rbcc` | Red blood cell count | millions/cmm |

**Comorbidities / symptoms (nominal)**

| Column | Meaning | Values |
|---|---|---|
| `htn` | Hypertension | yes / no |
| `dm` | Diabetes mellitus | yes / no |
| `cad` | Coronary artery disease | yes / no |
| `appet` | Appetite | good / poor |
| `pe` | Pedal edema (leg/ankle swelling) | yes / no |
| `ane` | Anemia | yes / no |

**Target**

| Column | Meaning | Values |
|---|---|---|
| `class` | Diagnosis | ckd (250 patients) / notckd (150 patients) |

# **Data Loading**

In [3]:
data, meta = arff.loadarff('chronic_kidney_disease_clean.arff')
df_CKD = pd.DataFrame(data)

for col in df_CKD.select_dtypes([object]).columns:
    df_CKD[col] = df_CKD[col].str.decode('utf-8')

df_CKD = df_CKD.drop_duplicates()

df_CKD

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1,0,?,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4,0,?,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2,3,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4,0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2,0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0,0,normal,normal,notpresent,notpresent,140.0,...,47.0,6700.0,4.9,no,no,no,good,no,no,notckd
396,42.0,70.0,1.025,0,0,normal,normal,notpresent,notpresent,75.0,...,54.0,7800.0,6.2,no,no,no,good,no,no,notckd
397,12.0,80.0,1.020,0,0,normal,normal,notpresent,notpresent,100.0,...,49.0,6600.0,5.4,no,no,no,good,no,no,notckd
398,17.0,60.0,1.025,0,0,normal,normal,notpresent,notpresent,114.0,...,51.0,7200.0,5.9,no,no,no,good,no,no,notckd


In [4]:
df_CKD.duplicated().sum()

np.int64(0)

In [5]:
for col in df_CKD.select_dtypes(include='object').columns:
    df_CKD[col] = df_CKD[col].replace('?', pd.NA)

# **Train-Test Split (Anti-Leakeage)**

In [6]:
numeric_cols = ['age','bp','bgr','bu','sc','sod','pot','hemo','pcv','wbcc','rbcc']
categorical_cols = ['sg','al','su','rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane']

X = df_CKD.drop(columns=['class'])
y = df_CKD['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# **EDA**

In [7]:
train_df = X_train.copy()
train_df['class'] = y_train

for col in numeric_cols:
    print(col, 'mean=', train_df[col].mean().round(2),
              'median=', train_df[col].median(),
              'skew=', train_df[col].skew().round(2))

for col in ['sc','bu','pot','sod','bgr','wbcc']:
    q1, q3 = train_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (train_df[col] < q1 - 1.5*iqr) | (train_df[col] > q3 + 1.5*iqr)
    print(col, mask.sum(), train_df.loc[mask, 'class'].value_counts())

age mean= 51.1 median= 54.0 skew= -0.61
bp mean= 76.32 median= 80.0 skew= 0.7
bgr mean= 149.34 median= 120.0 skew= 2.03
bu mean= 55.72 median= 41.0 skew= 2.87
sc mean= 2.9 median= 1.2 skew= 5.03
sod mean= 137.77 median= 138.0 skew= -1.09
pot mean= 4.66 median= 4.35 skew= 10.51
hemo mean= 12.7 median= 12.95 skew= -0.32
pcv mean= 39.42 median= 41.0 skew= -0.39
wbcc mean= 8345.02 median= 8000.0 skew= 1.77
rbcc mean= 4.78 median= 4.8 skew= -0.12
sc 39 class
ckd    39
Name: count, dtype: int64
bu 30 class
ckd    30
Name: count, dtype: int64
pot 4 class
ckd    4
Name: count, dtype: int64
sod 12 class
ckd    12
Name: count, dtype: int64
bgr 35 class
ckd    35
Name: count, dtype: int64
wbcc 9 class
ckd    9
Name: count, dtype: int64


### EDA Finding: Outlier Distribution
* **Observation:** 100% of statistical outliers (via IQR) belong exclusively to patients diagnosed with CKD.
* **Clinical Insight:** In renal biology, extreme toxic waste accumulation is a definitive signature of advanced kidney failure. Healthy (`notckd`) patients do not display these values. Tree-based models will leverage these sharp boundaries for near-perfect classification.

In [8]:
df_CKD.describe()

,age,bp,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc
count,391.000000,388.000000,356.000000,381.000000,383.000000,313.000000,312.000000,348.000000,329.000000,294.000000,269.000000
mean,51.483376,76.469072,148.036517,57.425722,3.072454,137.528754,4.627244,12.526437,38.884498,8406.122449,4.707435
std,17.169714,13.683637,79.281714,50.503006,5.741126,10.408752,3.193904,2.912587,8.990105,2944.474190,1.025323
min,2.000000,50.000000,22.000000,1.500000,0.400000,4.500000,2.500000,3.100000,9.000000,2200.000000,2.100000
25%,42.000000,70.000000,99.000000,27.000000,0.900000,135.000000,3.800000,10.300000,32.000000,6500.000000,3.900000
50%,55.000000,80.000000,121.000000,42.000000,1.300000,138.000000,4.400000,12.650000,40.000000,8000.000000,4.800000
75%,64.500000,80.000000,163.000000,66.000000,2.800000,142.000000,4.900000,15.000000,45.000000,9800.000000,5.400000
max,90.000000,180.000000,490.000000,391.000000,76.000000,163.000000,47.000000,17.800000,54.000000,26400.000000,8.000000


In [9]:
ckd_counts = train_df['class'].value_counts()
ckd_percentages = train_df['class'].value_counts(normalize=True) * 100

print("Target Class Imbalance Check:")
print(f"CKD:     {ckd_counts['ckd']} patients ({ckd_percentages['ckd']:.2f}%)")
print(f"No CKD:  {ckd_counts['notckd']} patients ({ckd_percentages['notckd']:.2f}%)\n")

imbalance_df = train_df['class'].value_counts().reset_index()
imbalance_df.columns = ['CKD Status', 'Patient Count']

imbalance_df['CKD Status'] = imbalance_df['CKD Status'].map({'ckd': 'CKD', 'notckd': 'No CKD'})

fig1 = px.bar(
    imbalance_df,
    x='CKD Status',
    y='Patient Count',
    color='CKD Status',

    color_discrete_map={
        'CKD': '#EF555B',
        'No CKD': '#636EFA'
    },

    category_orders={
        'CKD Status': ['CKD', 'No CKD']
    },
    title='Target Variable Distribution (CKD vs No CKD)',
    text_auto=True
)

fig1.update_layout(width=600, height=400, showlegend=False)
fig1.show()

Target Class Imbalance Check:
CKD:     200 patients (62.50%)
No CKD:  120 patients (37.50%)



In [10]:
key_features = ['sc', 'bu', 'hemo', 'pcv', 'al']
fig = make_subplots(rows=1, cols=len(key_features), subplot_titles=key_features)
for i, col in enumerate(key_features):
    plot_df = train_df.copy()
    if col == 'al':
        plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
    for cls in plot_df['class'].dropna().unique():
        fig.add_trace(go.Box(y=plot_df.loc[plot_df['class']==cls, col], name=cls, showlegend=(i==0)), row=1, col=i+1)
fig.update_layout(title='Key Clinical Markers by Class', height=400)
fig.show()

In [11]:
corr_df = train_df.copy()
corr_df['class_num'] = (corr_df['class'] == 'ckd').astype(int)
corr_matrix = corr_df[numeric_cols + ['class_num']].corr()

fig = px.imshow(corr_matrix, text_auto='.2f', color_continuous_scale='RdBu_r',
                 title='Feature Correlation Heatmap (Train Set)')
fig.show()

In [12]:
categorical_features = ['htn', 'dm', 'appet']
feature_labels = {'htn': 'Hypertension', 'dm': 'Diabetes Mellitus', 'appet': 'Appetite'}

for col in categorical_features:
    ct = pd.crosstab(train_df[col], train_df['class'], normalize='index') * 100
    ct = ct.reset_index().melt(id_vars=col, var_name='CKD Status', value_name='Percentage')
    ct['CKD Status'] = ct['CKD Status'].map({'ckd': 'CKD', 'notckd': 'No CKD'})

    fig = px.bar(
        ct,
        x=col,
        y='Percentage',
        color='CKD Status',

        color_discrete_map={
            'CKD': '#EF555B',
            'No CKD': '#636EFA'
        },

        category_orders={
            'CKD Status': ['CKD', 'No CKD']
        },
        barmode='group',
        title=f'{feature_labels[col]} vs CKD Status (% within group)',
        text_auto='.1f'
    )

    fig.update_layout(
        width=600, height=400,
        xaxis_title=feature_labels[col],
        yaxis_title='Percentage (%)',
        legend_title_text='Status'
    )
    fig.show()

In [13]:
df_CKD.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     391 non-null    float64
 1   bp      388 non-null    float64
 2   sg      353 non-null    object 
 3   al      354 non-null    object 
 4   su      351 non-null    object 
 5   rbc     248 non-null    object 
 6   pc      335 non-null    object 
 7   pcc     396 non-null    object 
 8   ba      396 non-null    object 
 9   bgr     356 non-null    float64
 10  bu      381 non-null    float64
 11  sc      383 non-null    float64
 12  sod     313 non-null    float64
 13  pot     312 non-null    float64
 14  hemo    348 non-null    float64
 15  pcv     329 non-null    float64
 16  wbcc    294 non-null    float64
 17  rbcc    269 non-null    float64
 18  htn     398 non-null    object 
 19  dm      398 non-null    object 
 20  cad     398 non-null    object 
 21  appet   399 non-null    object 
 22  pe

In [14]:
print(df_CKD.isnull().sum())

age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64


# **Preprocessing & Feature Engineering**

In [15]:
age_bins = [0, 12, 25, 45, 65, 120]
age_labels = ['Child','Youth','Young_Adult','Middle_Aged','Senior']
X_train['age_group'] = pd.cut(X_train['age'], bins=age_bins, labels=age_labels)
X_test['age_group'] = pd.cut(X_test['age'], bins=age_bins, labels=age_labels)

In [16]:
numeric_imputer = SimpleImputer(strategy='median')
X_train[numeric_cols] = numeric_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = numeric_imputer.transform(X_test[numeric_cols])

In [17]:
for col in ['sg','al','su']:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

In [18]:
X_train = X_train.drop(columns=['age_group'])
X_test = X_test.drop(columns=['age_group'])

In [19]:
scaler = StandardScaler()
continuous_cols = ['age','bp','bgr','bu','sc','sod','pot','hemo','pcv','wbcc','rbcc']
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [20]:
categorical_cols = ['sg','al','su','rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane']

X_train[categorical_cols] = X_train[categorical_cols].fillna(np.nan)
X_test[categorical_cols] = X_test[categorical_cols].fillna(np.nan)

cat_imputer = SimpleImputer(strategy='most_frequent')
cat_imputer.fit(X_train[categorical_cols])
X_train[categorical_cols] = cat_imputer.transform(X_train[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

In [21]:
categorical_cols = ['rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane']

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True, dtype=int)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
X_test = X_test[X_train.columns]

In [22]:
y_train = (y_train == 'ckd').astype(int)
y_test = (y_test == 'ckd').astype(int)

In [23]:
outlier_summary = {}

for col in ['sc', 'bu', 'pot', 'sod', 'bgr', 'wbcc']:
    q1, q3 = df_CKD[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_mask = (df_CKD[col] < q1 - 1.5*iqr) | (df_CKD[col] > q3 + 1.5*iqr)

    print(f"--- {col} ---")
    print("total outliers:", outlier_mask.sum())
    print(df_CKD.loc[outlier_mask, 'class'].value_counts())
    print()

--- sc ---
total outliers: 51
class
ckd    51
Name: count, dtype: int64

--- bu ---
total outliers: 38
class
ckd    38
Name: count, dtype: int64

--- pot ---
total outliers: 4
class
ckd    4
Name: count, dtype: int64

--- sod ---
total outliers: 16
class
ckd    16
Name: count, dtype: int64

--- bgr ---
total outliers: 34
class
ckd    34
Name: count, dtype: int64

--- wbcc ---
total outliers: 10
class
ckd    10
Name: count, dtype: int64



In [24]:
for col in ['sg','al','su']:
    X_train[col] = pd.to_numeric(X_train[col])
    X_test[col] = pd.to_numeric(X_test[col])

In [25]:
for col in df_CKD.select_dtypes(include='object').columns:
    print(col, df_CKD[col].unique())

sg ['1.020' '1.010' '1.005' '1.015' <NA> '1.025']
al ['1' '4' '2' '3' '0' <NA> '5']
su ['0' '3' '4' '1' <NA> '2' '5']
rbc [<NA> 'normal' 'abnormal']
pc ['normal' 'abnormal' <NA>]
pcc ['notpresent' 'present' <NA>]
ba ['notpresent' 'present' <NA>]
htn ['yes' 'no' <NA>]
dm ['yes' 'no' <NA>]
cad ['no' 'yes' <NA>]
appet ['good' 'poor' <NA>]
pe ['no' 'yes' <NA>]
ane ['no' 'yes' <NA>]
class ['ckd' 'notckd']


In [26]:
print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

0 0


# **Baseline Model Leaderboard**

In [27]:
baseline_metrics = {}

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
lin_pred_raw = lin_reg.predict(X_test)

y_pred_lin = np.where(lin_pred_raw >= 0.5, 1, 0)

baseline_metrics['Linear Regression'] = {
    'Accuracy': accuracy_score(y_test, y_pred_lin),
    'Precision': precision_score(y_test, y_pred_lin, zero_division=0),
    'Recall': recall_score(y_test, y_pred_lin, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_lin, zero_division=0)
}

base_rf = RandomForestClassifier(random_state=42)
base_rf.fit(X_train, y_train)
y_pred_base_rf = base_rf.predict(X_test)

baseline_metrics['Random Forest'] = {
    'Accuracy': accuracy_score(y_test, y_pred_base_rf),
    'Precision': precision_score(y_test, y_pred_base_rf, zero_division=0),
    'Recall': recall_score(y_test, y_pred_base_rf, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_base_rf, zero_division=0)
}

classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bagging': BaggingClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    baseline_metrics[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0)
    }

baseline_df = pd.DataFrame.from_dict(baseline_metrics, orient='index')

baseline_df = baseline_df.sort_values(by='F1-Score', ascending=False)

for score_col in baseline_df.columns:
    baseline_df[score_col] = (baseline_df[score_col] * 100).round(2).astype(str) + '%'

print("BASELINE MODEL PERFORMANCE LEADERBOARD:")
baseline_df

BASELINE MODEL PERFORMANCE LEADERBOARD:


,Accuracy,Precision,Recall,F1-Score
Random Forest,98.75%,98.04%,100.0%,99.01%
AdaBoost,98.75%,98.04%,100.0%,99.01%
SVM,98.75%,100.0%,98.0%,98.99%
XGBoost,97.5%,96.15%,100.0%,98.04%
Gradient Boosting,97.5%,96.15%,100.0%,98.04%
Bagging,97.5%,96.15%,100.0%,98.04%
Logistic Regression,97.5%,98.0%,98.0%,98.0%
Linear Regression,97.5%,100.0%,96.0%,97.96%
Naive Bayes,97.5%,100.0%,96.0%,97.96%
Decision Tree,96.25%,94.34%,100.0%,97.09%


# **Model Tuning**

In [28]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
search.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=RandomForestClassifier(random_state=42), n_iter=20,
                   n_jobs=-1,
                   param_distributions={'max_depth': [3, 5, 10, None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='f1')

# **Final Model Performance & Feature Importance**

In [29]:
print("Best hyperparameters found:")
print(search.best_params_)

print(f"\nBest cross-validated F1 (on train folds): {search.best_score_ * 100:.2f}%")

tuned_rf = search.best_estimator_
y_pred_tuned = tuned_rf.predict(X_test)

print("\nTuned Random Forest — performance on held-out test set:")

report = classification_report(
    y_test,
    y_pred_tuned,
    target_names=['No CKD', 'CKD'],
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

for col in ['precision', 'recall', 'f1-score']:
    report_df[col] = (
        report_df[col] * 100
    ).round(2).astype(str) + '%'

report_df['support'] = report_df['support'].astype(int)

display(report_df)

Best hyperparameters found:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}

Best cross-validated F1 (on train folds): 99.51%

Tuned Random Forest — performance on held-out test set:


,precision,recall,f1-score,support
No CKD,100.0%,93.33%,96.55%,30
CKD,96.15%,100.0%,98.04%,50
accuracy,97.5%,97.5%,97.5%,0
macro avg,98.08%,96.67%,97.3%,80
weighted avg,97.6%,97.5%,97.48%,80


In [30]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': tuned_rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='Feature Importance — Tuned Random Forest (All Features)',
    text='Importance',
    color='Importance',
    color_continuous_scale='Purples'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(
    height=700,
    yaxis_title='',
    xaxis_title='Importance',
    margin=dict(l=120, r=60, t=60, b=40)
)
fig.show()

In [31]:
comparison_df = pd.DataFrame({
    'Metric': [
        'Cross-validated F1',
        'Test Accuracy',
        'Test Precision',
        'Test Recall',
        'Test F1'
    ],
    'Score': [
        search.best_score_ * 100,
        accuracy_score(y_test, y_pred_tuned) * 100,
        precision_score(y_test, y_pred_tuned, zero_division=0) * 100,
        recall_score(y_test, y_pred_tuned, zero_division=0) * 100,
        f1_score(y_test, y_pred_tuned, zero_division=0) * 100
    ]
})

fig = px.bar(
    comparison_df,
    x='Metric',
    y='Score',
    text='Score',
    title='Tuned Random Forest — Validation and Test Performance',
    labels={
        'Metric': 'Evaluation metric',
        'Score': 'Performance (%)'
    }
)

fig.update_traces(
    texttemplate='%{text:.2f}%',
    textposition='outside',
    textfont_size=12
)

fig.update_yaxes(
    range=[0, 105],
    ticksuffix='%'
)

fig.update_layout(
    margin=dict(t=100, b=90, l=60, r=40)
)

fig.show()

In [32]:
baseline_metrics['Random Forest (Tuned)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned, zero_division=0),
    'Recall': recall_score(y_test, y_pred_tuned, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_tuned, zero_division=0)
}

baseline_df = pd.DataFrame.from_dict(baseline_metrics, orient='index')
baseline_df = baseline_df.sort_values(by='F1-Score', ascending=False)

for score_col in baseline_df.columns:
    baseline_df[score_col] = (baseline_df[score_col] * 100).round(2).astype(str) + '%'

print("BASELINE MODEL PERFORMANCE LEADERBOARD:")
baseline_df

BASELINE MODEL PERFORMANCE LEADERBOARD:


,Accuracy,Precision,Recall,F1-Score
Random Forest,98.75%,98.04%,100.0%,99.01%
AdaBoost,98.75%,98.04%,100.0%,99.01%
SVM,98.75%,100.0%,98.0%,98.99%
Bagging,97.5%,96.15%,100.0%,98.04%
Random Forest (Tuned),97.5%,96.15%,100.0%,98.04%
XGBoost,97.5%,96.15%,100.0%,98.04%
Gradient Boosting,97.5%,96.15%,100.0%,98.04%
Logistic Regression,97.5%,98.0%,98.0%,98.0%
Linear Regression,97.5%,100.0%,96.0%,97.96%
Naive Bayes,97.5%,100.0%,96.0%,97.96%


In [33]:
best_model_name = 'Random Forest (Tuned)'
best_model = tuned_rf

y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

fig = px.imshow(
    cm,
    text_auto=True,
    x=['Predicted No CKD', 'Predicted CKD'],
    y=['Actual No CKD', 'Actual CKD'],
    color_continuous_scale='Blues',
    title=f'Confusion Matrix — {best_model_name}'
)
fig.show()

print(f"Final selected model: {best_model_name}")
print("\nPerformance on held-out test set:")

report = classification_report(
    y_test,
    y_pred,
    target_names=['No CKD', 'CKD'],
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

for col in ['precision', 'recall', 'f1-score']:
    report_df[col] = (report_df[col] * 100).round(2).astype(str) + '%'

report_df['support'] = report_df['support'].astype(int)

display(report_df)

Final selected model: Random Forest (Tuned)

Performance on held-out test set:


,precision,recall,f1-score,support
No CKD,100.0%,93.33%,96.55%,30
CKD,96.15%,100.0%,98.04%,50
accuracy,97.5%,97.5%,97.5%,0
macro avg,98.08%,96.67%,97.3%,80
weighted avg,97.6%,97.5%,97.48%,80
